In [1]:
import pandas as pd
import numpy as np

# Temperature and Water Merge

In [2]:
# import data
bus = pd.read_csv("../Data/Bus_Routes.csv", low_memory = False)
l_station = pd.read_csv("../Data/L_Stations.csv")

In [3]:
# TMAX	Maximum temperature
# TMIN	Minimum temperature
# TOBS	Temperature at the time of observation
temp = pd.read_csv("../Data/Temperature.csv")
temp = temp.drop(columns = ['TMAX_ATTRIBUTES', 'TMIN_ATTRIBUTES', 'TOBS_ATTRIBUTES'])
temp.head()

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,TMAX,TMIN,TOBS
0,USC00111550,"CHICAGO NORTHERLY ISLAND, IL US",41.8558,-87.6094,177.7,2005-06-01,68.0,55.0,61.0
1,USC00111550,"CHICAGO NORTHERLY ISLAND, IL US",41.8558,-87.6094,177.7,2005-06-02,64.0,55.0,58.0
2,USC00111550,"CHICAGO NORTHERLY ISLAND, IL US",41.8558,-87.6094,177.7,2005-06-03,75.0,56.0,59.0
3,USC00111550,"CHICAGO NORTHERLY ISLAND, IL US",41.8558,-87.6094,177.7,2005-06-04,75.0,58.0,74.0
4,USC00111550,"CHICAGO NORTHERLY ISLAND, IL US",41.8558,-87.6094,177.7,2005-06-05,87.0,64.0,72.0


In [4]:
# PRCP	Precipitation
# SNOW	Snowfall
# SNWD	Snow depth
water = pd.read_csv("../Data/Water.csv")
water = water.drop(columns=['DAPR', 'DAPR_ATTRIBUTES', 'MDPR', 'MDPR_ATTRIBUTES', 'PRCP_ATTRIBUTES', 'SNOW_ATTRIBUTES', 'SNWD_ATTRIBUTES', 'WESD', 'WESD_ATTRIBUTES', 'WESF', 'WESF_ATTRIBUTES'])
water = water.fillna(0)
water.head()

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,PRCP,SNOW,SNWD
0,US1ILCK0192,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,192.6,2011-08-31,0.06,0.0,0.0
1,US1ILCK0192,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,192.6,2011-09-01,0.00,0.0,0.0
2,US1ILCK0192,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,192.6,2011-09-02,0.00,0.0,0.0
3,US1ILCK0192,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,192.6,2011-09-03,0.00,0.0,0.0
4,US1ILCK0192,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,192.6,2011-09-04,0.04,0.0,0.0


In [5]:
# TMAX	Maximum temperature
# TMIN	Minimum temperature
# TOBS	Temperature at the time of observation
# PRCP	Precipitation
# SNOW	Snowfall
# SNWD	Snow depth
weather = pd.merge(left = temp, right = water, how = "inner", on = "DATE", suffixes = ('_temp', '_water'))
weather.head()

,STATION_temp,NAME_temp,LATITUDE_temp,LONGITUDE_temp,ELEVATION_temp,DATE,TMAX,TMIN,TOBS,STATION_water,NAME_water,LATITUDE_water,LONGITUDE_water,ELEVATION_water,PRCP,SNOW,SNWD
0,USC00111550,"CHICAGO NORTHERLY ISLAND, IL US",41.85580,-87.609400,177.7,2011-08-31,90.0,67.0,78.0,US1ILCK0192,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,192.6,0.06,0.0,0.0
1,US1ILCK0192,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,192.6,2011-08-31,NaN,NaN,NaN,US1ILCK0192,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,192.6,0.06,0.0,0.0
2,USC00111550,"CHICAGO NORTHERLY ISLAND, IL US",41.85580,-87.609400,177.7,2011-09-01,97.0,75.0,88.0,US1ILCK0192,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,192.6,0.00,0.0,0.0
3,US1ILCK0192,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,192.6,2011-09-01,NaN,NaN,NaN,US1ILCK0192,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,192.6,0.00,0.0,0.0
4,USC00111550,"CHICAGO NORTHERLY ISLAND, IL US",41.85580,-87.609400,177.7,2011-09-02,96.0,77.0,78.0,US1ILCK0192,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,192.6,0.00,0.0,0.0


# L line and Bus Merge

In [ ]:
def clean_transit(df, mode, id_col, name_col=None):
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"], format="%m/%d/%Y")
    out["rides"] = out["rides"].astype(str).str.replace(",", "").astype(int)   # "6,217" -> 6217
    out["mode"] = mode
    out["entity_id"] = out[id_col].astype(str)
    out["entity_name"] = out[name_col] if name_col else out[id_col].astype(str)
    return out[["mode", "entity_id", "entity_name", "date", "daytype", "rides"]]

bus_c = clean_transit(bus, "bus", "route")
l_c   = clean_transit(l_station, "L", "station_id", "stationname")

# station 40140 appears under two names (Skokie / Dempster-Skokie) keep the most recent name
latest_name = l_c.sort_values("date").groupby("entity_id")["entity_name"].last()
l_c["entity_name"] = l_c["entity_id"].map(latest_name)

# 618 L rows are duplicated station/date pairs with slightly different counts (all 2011), average them
l_c = (l_c.groupby(["mode", "entity_id", "entity_name", "date", "daytype"], as_index=False)["rides"]
          .mean())

In [7]:
# merge bus and L data into a single dataframe
transit = pd.concat([bus_c, l_c], ignore_index=True)
transit["year"]  = transit["date"].dt.year
transit["month"] = transit["date"].dt.month
transit["dow"]   = transit["date"].dt.day_name()
transit["daytype"] = transit["daytype"].map({"W": "Weekday", "A": "Saturday", "U": "Sunday/Holiday"})

## Transit and Weather Merge (keep stations separate)

In [ ]:
# THIS MERGING KEEPS THE STATION IDENTITY, SO YOU CAN TELL THE TWO ROWS APART
weather_merge = weather.copy()
weather_merge["DATE"] = pd.to_datetime(weather_merge["DATE"])

# Keep the station identity so you can tell the two rows apart
weather_merge = weather_merge.rename(columns={
    "NAME_temp": "temp_station",
    "LATITUDE_temp": "temp_lat",
    "LONGITUDE_temp": "temp_lon",
})[["DATE", "temp_station", "temp_lat", "temp_lon", "TMAX", "TMIN", "TOBS", "PRCP", "SNOW", "SNWD"]]

transit_weather = transit.merge(
    weather_merge,
    how="inner",
    left_on="date",
    right_on="DATE",
    validate="many_to_many",   # most dates have 2 weather rows, so each transit row repeats
).drop(columns="DATE")

In [10]:
#double check sizes

print(transit.shape, transit_weather.shape)
print(transit_weather.duplicated(["mode", "entity_id", "date", "temp_station"]).sum(), "duplicates (should be 0)")
transit_weather.head()

(2461407, 9) (2260933, 18)
0 duplicates (should be 0)


,mode,entity_id,entity_name,date,daytype,rides,year,month,dow,temp_station,temp_lat,temp_lon,TMAX,TMIN,TOBS,PRCP,SNOW,SNWD
0,bus,1,1,2015-04-25,Saturday,438.0,2015,4,Saturday,"CHICAGO NORTHERLY ISLAND, IL US",41.85580,-87.609400,45.0,40.0,41.0,0.1,0.0,0.0
1,bus,1,1,2015-04-25,Saturday,438.0,2015,4,Saturday,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,NaN,NaN,NaN,0.1,0.0,0.0
2,bus,10,10,2015-04-25,Saturday,986.0,2015,4,Saturday,"CHICAGO NORTHERLY ISLAND, IL US",41.85580,-87.609400,45.0,40.0,41.0,0.1,0.0,0.0
3,bus,10,10,2015-04-25,Saturday,986.0,2015,4,Saturday,"PARK RIDGE 0.7 WNW, IL US",42.01676,-87.855668,NaN,NaN,NaN,0.1,0.0,0.0
4,bus,1001,1001,2015-04-25,Saturday,3.0,2015,4,Saturday,"CHICAGO NORTHERLY ISLAND, IL US",41.85580,-87.609400,45.0,40.0,41.0,0.1,0.0,0.0


## Transit and Weather Merge (combine stations)

In [11]:
# THIS MERGE MERGES THE STATION IDENTITY, FOR MAPPING PURPOSES SO THAT THERE ARE NO REPEATS
weather_daily = (weather.assign(DATE=pd.to_datetime(weather["DATE"]))
                        .groupby("DATE", as_index=False)
                        .agg(TMAX=("TMAX", "mean"), TMIN=("TMIN", "mean"), TOBS=("TOBS", "mean"),
                             PRCP=("PRCP", "first"), SNOW=("SNOW", "first"), SNWD=("SNWD", "first")))

transit_weather_merged = transit.merge(weather_daily, how="inner", left_on="date", right_on="DATE",
                                validate="many_to_one").drop(columns="DATE")

In [13]:
# check sizes
print(transit.shape, transit_weather_merged.shape)
print(transit_weather_merged.duplicated(["mode","entity_id","date"]).sum(), "duplicate rows (should be 0)")
transit_weather_merged.head()

(2461407, 9) (1154613, 15)
0 duplicate rows (should be 0)


,mode,entity_id,entity_name,date,daytype,rides,year,month,dow,TMAX,TMIN,TOBS,PRCP,SNOW,SNWD
0,bus,1,1,2015-04-25,Saturday,438.0,2015,4,Saturday,45.0,40.0,41.0,0.1,0.0,0.0
1,bus,10,10,2015-04-25,Saturday,986.0,2015,4,Saturday,45.0,40.0,41.0,0.1,0.0,0.0
2,bus,1001,1001,2015-04-25,Saturday,3.0,2015,4,Saturday,45.0,40.0,41.0,0.1,0.0,0.0
3,bus,103,103,2015-04-25,Saturday,1163.0,2015,4,Saturday,45.0,40.0,41.0,0.1,0.0,0.0
4,bus,106,106,2015-04-25,Saturday,489.0,2015,4,Saturday,45.0,40.0,41.0,0.1,0.0,0.0
